In [1]:
# أهمية الخلية:
# التأكد من توفر مكتبات التعامل مع البيانات الجغرافية التي سنستخدمها
# لاستخراج ومعالجة بيانات مدينة القاهرة من OpenStreetMap.
#
# GeoPandas: للتعامل مع البيانات الجغرافية والجداول المكانية.
# OSMnx: لاستخراج بيانات OpenStreetMap مثل الحدود والطرق ومحطات النقل.
# Shapely: للتعامل مع الأشكال والهندسة الجغرافية.

%pip install geopandas osmnx shapely

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# أهمية الخلية:
# التأكد من أن مكتبات GIS الأساسية تعمل بشكل صحيح بعد تثبيتها،
# قبل البدء في تحميل أي بيانات جغرافية من OpenStreetMap.
# هذه الخطوة تمنعنا من بدء تحميل البيانات ثم اكتشاف وجود مشكلة في البيئة.

import geopandas as gpd
import osmnx as ox
import shapely

print("GeoPandas version:", gpd.__version__)
print("OSMnx version:", ox.__version__)
print("Shapely version:", shapely.__version__)

GeoPandas version: 1.1.4
OSMnx version: 2.1.1
Shapely version: 2.1.2


In [2]:
# أهمية الخلية:
# الحصول على الحدود الجغرافية للقاهرة من OpenStreetMap،
# عشان نحدد النطاق المكاني اللي هنشتغل عليه قبل استخراج باقي البيانات.
# هنطبع المعلومات بالإنجليزي لتكون الـ output موحدة وسهلة في المشروع.

import osmnx as ox

cairo = ox.geocode_to_gdf("Cairo Governorate, Egypt")

print("Number of results:", len(cairo))

print("\nAvailable columns:")
print(cairo.columns.tolist())

print("\nLocation information:")
print(cairo[["name", "display_name", "geometry"]].to_string())

Number of results: 1

Available columns:
['geometry', 'bbox_west', 'bbox_south', 'bbox_east', 'bbox_north', 'place_id', 'osm_type', 'osm_id', 'lat', 'lon', 'class', 'type', 'place_rank', 'importance', 'addresstype', 'name', 'display_name']

Location information:
    name  display_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [3]:
# أهمية الخلية:
# فحص نظام الإحداثيات (CRS) وحدود المنطقة والمساحة التقريبية للقاهرة.
# الخطوة دي مهمة للتأكد من أن الـ Polygon صالح للاستخدام في التحليل المكاني
# قبل استخراج طبقات الطرق والخدمات والمواصلات.

print("Coordinate Reference System (CRS):", cairo.crs)

print("\nBounding Box:")
print(cairo.total_bounds)

# تحويل النظام إلى CRS مناسب للحسابات المترية
cairo_projected = cairo.to_crs("EPSG:32636")

area_km2 = cairo_projected.geometry.area.sum() / 1_000_000

print("\nApproximate Area (km²):", round(area_km2, 2))

Coordinate Reference System (CRS): epsg:4326

Bounding Box:
[31.214555  29.7483062 31.9090054 30.3209168]

Approximate Area (km²): 3006.45


In [4]:
# أهمية الخلية:
# استخراج شبكة الطرق داخل حدود القاهرة من OpenStreetMap.
# شبكة الطرق مهمة جدًا في UrbanMind AI لأنها هتساعدنا لاحقًا
# في تحليل توزيع الطرق وربط البيانات الجغرافية ببيانات Transportation وTraffic.

import osmnx as ox

# استخراج شبكة الطرق داخل حدود القاهرة
G = ox.graph_from_polygon(
    cairo.geometry.iloc[0],
    network_type="drive",
    simplify=True
)

print("Road network extracted successfully.")

print("\nNumber of nodes:", len(G.nodes))
print("Number of edges:", len(G.edges))

Road network extracted successfully.

Number of nodes: 137661
Number of edges: 322731


In [5]:
# أهمية الخلية:
# تحويل شبكة الطرق من Network Graph إلى جداول جغرافية منظمة.
# الـ Nodes تمثل نقاط الشبكة، والـ Edges تمثل مقاطع الطرق.
# ده هيسمح لنا لاحقًا بتحليل الطرق، حساب الأطوال، وعمل Spatial Analysis.

import osmnx as ox

nodes, edges = ox.graph_to_gdfs(G)

print("Road network converted successfully.")

print("\nNodes:")
print("Number of nodes:", len(nodes))
print("Number of columns:", len(nodes.columns))

print("\nEdges:")
print("Number of edges:", len(edges))
print("Number of columns:", len(edges.columns))

print("\nNode columns:")
print(nodes.columns.tolist())

print("\nEdge columns:")
print(edges.columns.tolist())

Road network converted successfully.

Nodes:
Number of nodes: 137661
Number of columns: 8

Edges:
Number of edges: 322731
Number of columns: 15

Node columns:
['y', 'x', 'street_count', 'highway', 'ref', 'junction', 'railway', 'geometry']

Edge columns:
['osmid', 'highway', 'junction', 'oneway', 'reversed', 'length', 'geometry', 'lanes', 'maxspeed', 'name', 'ref', 'bridge', 'access', 'tunnel', 'width']


In [6]:
# أهمية الخلية:
# فحص جودة بيانات الطرق قبل التنظيف.
# هنحسب عدد ونسبة القيم المفقودة في أهم أعمدة Road Network،
# عشان نعرف البيانات المتاحة فعليًا من OpenStreetMap
# قبل اتخاذ أي قرار بحذف أو معالجة القيم الناقصة.

important_columns = [
    "highway",
    "length",
    "name",
    "lanes",
    "maxspeed",
    "oneway"
]

quality_check = edges[important_columns].isna().sum().to_frame("Missing Values")

quality_check["Missing Percentage"] = (
    quality_check["Missing Values"] / len(edges) * 100
).round(2)

quality_check["Available Values"] = (
    len(edges) - quality_check["Missing Values"]
)

print("Road Network Data Quality Check")
print("\nTotal road segments:", len(edges))

print("\nMissing values by column:")
print(quality_check)

Road Network Data Quality Check

Total road segments: 322731

Missing values by column:
          Missing Values  Missing Percentage  Available Values
highway                0                0.00            322731
length                 0                0.00            322731
name              111628               34.59            211103
lanes             318947               98.83              3784
maxspeed          319331               98.95              3400
oneway                 0                0.00            322731


In [7]:
# أهمية الخلية:
# معرفة أنواع الطرق الموجودة فعليًا داخل شبكة القاهرة،
# وعدد مقاطع الطرق لكل نوع.
# لن نحذف أو نعدل أي نوع في هذه المرحلة؛ الهدف فقط فهم البيانات
# قبل اتخاذ أي قرار متعلق بالتنظيف أو التحليل.

highway_types = (
    edges["highway"]
    .astype(str)
    .value_counts()
    .rename_axis("Highway Type")
    .reset_index(name="Number of Segments")
)

print("Highway Types in Cairo Road Network")
print("\nTotal unique highway types:", len(highway_types))

print("\nHighway type distribution:")
print(highway_types.to_string(index=False))

Highway Types in Cairo Road Network

Total unique highway types: 42

Highway type distribution:
                      Highway Type  Number of Segments
                       residential              263671
                          tertiary               25242
                         secondary                8209
                      unclassified                7913
                           primary                6190
                     tertiary_link                3612
                             trunk                2286
                    secondary_link                1691
                      primary_link                1502
                        trunk_link                 973
                          motorway                 621
                     motorway_link                 554
                     living_street                 128
     ['tertiary_link', 'tertiary']                  18
       ['residential', 'tertiary']                  15
        ['trunk', 'motor

In [8]:
# أهمية الخلية:
# عرض جميع أنواع الطرق الـ 42 الموجودة في البيانات بدون اقتطاع الـ output.
# الخطوة دي مهمة لفهم الـ Road Network بالكامل قبل أي تنظيف أو تصنيف.
# لن نقوم بتعديل أو حذف أي بيانات في هذه المرحلة.

for i, row in highway_types.iterrows():
    print(f"{i + 1}. {row['Highway Type']} -> {row['Number of Segments']} segments")

print("\nTotal unique highway types:", len(highway_types))
print("Total road segments:", len(edges))

1. residential -> 263671 segments
2. tertiary -> 25242 segments
3. secondary -> 8209 segments
4. unclassified -> 7913 segments
5. primary -> 6190 segments
6. tertiary_link -> 3612 segments
7. trunk -> 2286 segments
8. secondary_link -> 1691 segments
9. primary_link -> 1502 segments
10. trunk_link -> 973 segments
11. motorway -> 621 segments
12. motorway_link -> 554 segments
13. living_street -> 128 segments
14. ['tertiary_link', 'tertiary'] -> 18 segments
15. ['residential', 'tertiary'] -> 15 segments
16. ['trunk', 'motorway_link'] -> 11 segments
17. ['secondary', 'secondary_link'] -> 8 segments
18. ['unclassified', 'residential'] -> 8 segments
19. ['tertiary', 'tertiary_link'] -> 7 segments
20. ['primary_link', 'primary'] -> 6 segments
21. ['secondary', 'tertiary'] -> 6 segments
22. ['secondary_link', 'tertiary'] -> 5 segments
23. ['unclassified', 'trunk_link'] -> 5 segments
24. ['secondary', 'trunk_link'] -> 5 segments
25. ['trunk_link', 'primary'] -> 4 segments
26. ['secondary', 're

In [9]:
# أهمية الخلية:
# فحص شكل القيم الموجودة في عمود highway قبل إنشاء أي تصنيف جديد.
# بعض القيم عبارة عن نوع طريق واحد، وبعضها عبارة عن أكثر من نوع داخل List.
# لازم نفهم نسبتها أولًا حتى لا نغيّر أو نفسّر بيانات OpenStreetMap بشكل خاطئ.

import ast

def is_list_value(value):
    return isinstance(value, list)

list_highway_count = edges["highway"].apply(is_list_value).sum()
single_highway_count = len(edges) - list_highway_count

print("Highway value structure analysis")

print("\nTotal road segments:", len(edges))
print("Single highway type:", single_highway_count)
print("Multiple highway types:", list_highway_count)

print(
    "\nPercentage with multiple highway types:",
    round(list_highway_count / len(edges) * 100, 2),
    "%"
)

Highway value structure analysis

Total road segments: 322731
Single highway type: 322593
Multiple highway types: 138

Percentage with multiple highway types: 0.04 %


In [10]:
# أهمية الخلية:
# إنشاء تصنيف مبسط للطرق Road Class مع الحفاظ على عمود highway الأصلي.
# التصنيف هيسهّل التحليل والـ visualization لاحقًا،
# بينما highway سيظل محتفظًا بقيمة OpenStreetMap الأصلية بدون تعديل.

def classify_road(highway):
    # التعامل مع القيم التي تحتوي على أكثر من نوع طريق
    if isinstance(highway, list):
        highway = highway[0]

    highway = str(highway)

    if highway in ["motorway", "motorway_link"]:
        return "motorway"
    elif highway in ["trunk", "trunk_link"]:
        return "trunk"
    elif highway in ["primary", "primary_link"]:
        return "primary"
    elif highway in ["secondary", "secondary_link"]:
        return "secondary"
    elif highway in ["tertiary", "tertiary_link"]:
        return "tertiary"
    elif highway in ["residential", "living_street"]:
        return "residential"
    else:
        return "other"


edges["road_class"] = edges["highway"].apply(classify_road)

print("Road classification created successfully.")

print("\nRoad class distribution:")
print(
    edges["road_class"]
    .value_counts()
    .rename_axis("Road Class")
    .to_string()
)

Road classification created successfully.

Road class distribution:
Road Class
residential    263814
tertiary        28879
secondary        9939
other            7934
primary          7707
trunk            3281
motorway         1177


In [11]:
# أهمية الخلية:
# إنشاء نسخة مخصصة للتحليل من بيانات الطرق مع الاحتفاظ بالبيانات الأصلية.
# هنختار أهم الأعمدة التي نحتاجها في UrbanMind AI،
# مع الإبقاء على geometry لأنها أساسية لأي Spatial Analysis لاحقًا.

roads = edges[
    [
        "osmid",
        "highway",
        "road_class",
        "name",
        "length",
        "lanes",
        "maxspeed",
        "oneway",
        "geometry"
    ]
].copy()

print("Road analysis layer created successfully.")

print("\nNumber of road segments:", len(roads))
print("Number of columns:", len(roads.columns))

print("\nColumns:")
print(roads.columns.tolist())

print("\nCoordinate Reference System:")
print(roads.crs)

Road analysis layer created successfully.

Number of road segments: 322731
Number of columns: 9

Columns:
['osmid', 'highway', 'road_class', 'name', 'length', 'lanes', 'maxspeed', 'oneway', 'geometry']

Coordinate Reference System:
epsg:4326


In [12]:
# أهمية الخلية:
# التأكد من سلامة Road Layer قبل حفظها واستخدامها في UrbanMind AI.
# هنراجع عدد القيم المفقودة، والقيم السالبة أو الصفرية في طول الطرق،
# ونتأكد أن الـ geometry كلها موجودة وصالحة.

print("Final Road Layer Quality Check")

print("\nMissing values:")
print(roads.isna().sum())

print("\nInvalid road lengths:")
print("Zero length:", (roads["length"] == 0).sum())
print("Negative length:", (roads["length"] < 0).sum())

print("\nGeometry check:")
print("Missing geometry:", roads.geometry.isna().sum())
print("Invalid geometry:", (~roads.geometry.is_valid).sum())

print("\nCRS:", roads.crs)
print("Total road segments:", len(roads))


Final Road Layer Quality Check

Missing values:
osmid              0
highway            0
road_class         0
name          111628
length             0
lanes         318947
maxspeed      319331
oneway             0
geometry           0
dtype: int64

Invalid road lengths:
Zero length: 0
Negative length: 0

Geometry check:
Missing geometry: 0
Invalid geometry: 0

CRS: epsg:4326
Total road segments: 322731


In [13]:
import sys

print("Python environment:")
print(sys.executable)

!{sys.executable} -m pip install fiona

Python environment:
c:\Users\SH\AppData\Local\Programs\Python\Python311\python.exe



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
# أهمية الخلية:
# تثبيت Fiona كـ GIS I/O engine بديل لـ pyogrio،
# لأن GeoPandas يحتاج محركًا خارجيًا لحفظ ملفات GeoPackage.
# لن نلمس بيانات الطرق نفسها في هذه الخطوة.

%pip install fiona

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import fiona

print("Fiona installed successfully!")
print("Fiona version:", fiona.__version__)

Fiona installed successfully!
Fiona version: 1.10.1


In [16]:
from pathlib import Path

geo_processed = Path(
    r"D:\Courses\MachineLearning\Depi\Final Project\data\geographic\processed"
)

geo_processed.mkdir(parents=True, exist_ok=True)

# حفظ Road Layer كـ GeoPackage باستخدام Fiona
roads_gpkg = geo_processed / "cairo_roads.gpkg"

roads.to_file(
    roads_gpkg,
    layer="roads",
    driver="GPKG",
    engine="fiona"
)

# حفظ نسخة CSV بدون geometry
roads_csv = geo_processed / "cairo_roads.csv"

roads.drop(columns="geometry").to_csv(
    roads_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Road Layer saved successfully!")

print("\nGeoPackage:")
print(roads_gpkg)

print("\nCSV:")
print(roads_csv)

print("\nRows saved:", len(roads))

Road Layer saved successfully!

GeoPackage:
D:\Courses\MachineLearning\Depi\Final Project\data\geographic\processed\cairo_roads.gpkg

CSV:
D:\Courses\MachineLearning\Depi\Final Project\data\geographic\processed\cairo_roads.csv

Rows saved: 322731
